In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv kailash-ml kailash

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 7: Workflow Orchestration, Model Registry, and
#                        Hyperparameter Search
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Build ML workflows using Kailash WorkflowBuilder with named nodes
#   - Understand the runtime.execute(workflow.build()) execution pattern
#   - Create custom workflow nodes with branching and conditional logic
#   - Persist structured ML results to a database using DataFlow @db.model
#   - Implement Bayesian hyperparameter search with HyperparameterSearch
#   - Register, version, and promote models through the ModelRegistry
#     lifecycle
#   - Define ModelSignature for input/output contract validation
#   - Query and compare persisted model evaluations across experiments
#
# PREREQUISITES:
#   - MLFP03 Exercises 1-6 (feature engineering through SHAP)
#   - MLFP02 complete (preprocessing pipeline, Singapore credit data)
#
# ESTIMATED TIME: ~150-180 minutes
#
# TASKS:
#   1.  Build a Kailash workflow for the ML pipeline
#   2.  Define @db.model for evaluation results (DataFlow)
#   3.  Train and evaluate manually (parallel to workflow)
#   4.  Persist results with db.express
#   5.  Bayesian hyperparameter search with HyperparameterSearch
#   6.  Build branching workflow with conditional execution
#   7.  Define ModelSignature (input/output schema)
#   8.  Register model in ModelRegistry
#   9.  Promote model through staging → production lifecycle
#   10. Query persisted results and compare experiments
#   11. Build complete orchestrated pipeline
#   12. Pipeline audit and reproducibility check
#
# DATASET: Singapore credit scoring (from MLFP02)
#   Target: default prediction (binary, 12% positive rate)
#   Goal: orchestrate the full pipeline as a reproducible Kailash workflow
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import os
import pickle

import numpy as np
import polars as pl
import lightgbm as lgb
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
)
from sklearn.model_selection import cross_val_score

from kailash.workflow.builder import WorkflowBuilder
from kailash.runtime import LocalRuntime

try:
    from kailash.dataflow import DataFlow, field

    HAS_DATAFLOW = True
except ImportError:
    try:
        from dataflow import DataFlow, field

        HAS_DATAFLOW = True
    except ImportError:
        HAS_DATAFLOW = False
        print("  Note: kailash-dataflow not available. Skipping DataFlow tasks.")

from kailash.db import ConnectionManager
from kailash_ml import PreprocessingPipeline, ModelVisualizer
from kailash_ml.interop import to_sklearn_input
from kailash_ml.engines.training_pipeline import TrainingPipeline, ModelSpec, EvalSpec
from kailash_ml.engines.experiment_tracker import ExperimentTracker

try:
    from kailash_ml.engines.hyperparameter_search import (
        HyperparameterSearch,
        SearchSpace,
        ParamDistribution,
        SearchConfig,
    )

    HAS_HYPERPARAM = True
except ImportError:
    HAS_HYPERPARAM = False
    print("  Note: HyperparameterSearch not available.")

try:
    from kailash_ml.engines.model_registry import ModelRegistry

    HAS_REGISTRY = True
except ImportError:
    HAS_REGISTRY = False
    print("  Note: ModelRegistry not available.")

from kailash_ml.types import ModelSignature, FeatureSchema, FeatureField

try:
    from kailash_ml.types import MetricSpec

    HAS_METRIC_SPEC = True
except ImportError:
    HAS_METRIC_SPEC = False

from shared import MLFPDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()



### Data Loading


In [ ]:
loader = MLFPDataLoader()
credit = loader.load("mlfp02", "sg_credit_scoring.parquet")



## TASK 1: Build a Kailash ML workflow


In [ ]:
# WorkflowBuilder: nodes, connections, runtime.execute(workflow.build())

# TODO: Create a WorkflowBuilder named "credit_scoring_pipeline"
workflow = ____  # Hint: WorkflowBuilder("credit_scoring_pipeline")

# Node 1: Data preprocessing
# TODO: Add a DataPreprocessNode named "preprocess" with data_source,
#       target, train_size, seed, normalize, categorical_encoding, imputation_strategy
workflow.add_node(
    "DataPreprocessNode",
    "preprocess",
    {
        "data_source": "sg_credit_scoring",
        "target": ____,  # Hint: "default"
        "train_size": ____,  # Hint: 0.8
        "seed": 42,
        "normalize": False,
        "categorical_encoding": ____,  # Hint: "ordinal"
        "imputation_strategy": ____,  # Hint: "median"
    },
)

# Node 2: Model training
workflow.add_node(
    "ModelTrainNode",
    "train",
    {
        "model_class": "lightgbm.LGBMClassifier",
        "hyperparameters": {
            "n_estimators": 500,
            "learning_rate": 0.1,
            "max_depth": 6,
            "scale_pos_weight": 7.3,
        },
    },
    connections=["preprocess"],
)

# Node 3: Evaluation
workflow.add_node(
    "ModelEvalNode",
    "evaluate",
    {"metrics": ["accuracy", "f1", "auc_roc", "auc_pr", "log_loss"]},
    connections=["train"],
)

# Node 4: Persist results
workflow.add_node(
    "PersistNode",
    "persist",
    {"storage": "sqlite:///mlfp03_models.db"},
    connections=["evaluate"],
)

# TODO: Create a LocalRuntime and execute the workflow
# Hint: runtime = LocalRuntime(); results, run_id = runtime.execute(workflow.build())
runtime = ____
print("=== Executing Workflow ===")
try:
    results, run_id = ____  # Hint: runtime.execute(workflow.build())
    print(f"Run ID: {run_id}")
    print(f"Node results: {list(results.keys())}")
    HAS_WORKFLOW = True
except Exception as e:
    print(f"  Note: Workflow execution failed ({type(e).__name__}: {e})")
    print("  Custom nodes require registration. Proceeding with manual pipeline.")
    results, run_id = {}, "manual-run"
    HAS_WORKFLOW = False



### Checkpoint 1


In [ ]:
assert results is not None, "Workflow should return results"
assert run_id is not None, "Workflow should return a run_id"
if HAS_WORKFLOW:
    assert len(results) > 0, "Workflow should produce node results"
    print("\n✓ Checkpoint 1 passed — workflow executed successfully\n")
else:
    print("\n⚠ Checkpoint 1 — workflow fallback active, continuing manually\n")
# INTERPRETATION: WorkflowBuilder captures the pipeline as a DAG.
# Each node receives inputs from connections and passes outputs downstream.
# LocalRuntime executes nodes sequentially in dependency order.



## TASK 2: Define @db.model for evaluation results


Stores evaluation results for trained models.


Stores model metadata and serialisation path.


In [ ]:
if HAS_DATAFLOW:
    # TODO: Create a DataFlow instance connected to "sqlite:///mlfp03_models.db"
    db = ____  # Hint: DataFlow("sqlite:///mlfp03_models.db")

    # TODO: Decorate with @db.model
    ____

    class ModelEvaluation:

        id: int = field(primary_key=True)
        model_name: str = field()
        dataset: str = field()
        accuracy: float = field()
        f1_score: float = field()
        auc_roc: float = field()
        auc_pr: float = field()
        log_loss_val: float = field()
        train_size: int = field()
        test_size: int = field()
        feature_count: int = field()
        hyperparameters: str = field(default="{}")

    # TODO: Decorate with @db.model
    ____

    class ModelArtifact:

        id: int = field(primary_key=True)
        model_name: str = field()
        version: int = field()
        artifact_path: str = field()
        is_production: bool = field(default=False)
        created_by: str = field(default="mlfp03")



### Checkpoint 2


In [ ]:
if HAS_DATAFLOW:
    assert ModelEvaluation is not None, "ModelEvaluation should be defined"
    assert ModelArtifact is not None, "ModelArtifact should be defined"
# INTERPRETATION: DataFlow's @db.model decorator registers your class as a
# database table.  Schema migrations happen automatically.
print("\n✓ Checkpoint 2 passed — DataFlow models defined\n")



## TASK 3: Train and evaluate manually


In [ ]:
pipeline = PreprocessingPipeline()
# TODO: Call pipeline.setup() with credit data, target="default", seed=42,
#       normalize=False, categorical_encoding="ordinal"
result = pipeline.setup(
    ____,  # Hint: the credit polars DataFrame
    target=____,  # Hint: "default"
    seed=42,
    normalize=False,
    categorical_encoding=____,  # Hint: "ordinal"
)

X_train, y_train, col_info = to_sklearn_input(
    result.train_data,
    feature_columns=[c for c in result.train_data.columns if c != "default"],
    target_column="default",
)
X_test, y_test, _ = to_sklearn_input(
    result.test_data,
    feature_columns=[c for c in result.test_data.columns if c != "default"],
    target_column="default",
)

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
    random_state=42,
    verbose=-1,
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# TODO: Compute all five metrics: accuracy, f1, auc_roc, auc_pr, log_loss
eval_metrics = {
    "accuracy": ____,  # Hint: accuracy_score(y_test, y_pred)
    "f1": ____,  # Hint: f1_score(y_test, y_pred)
    "auc_roc": ____,  # Hint: roc_auc_score(y_test, y_proba)
    "auc_pr": ____,  # Hint: average_precision_score(y_test, y_proba)
    "log_loss": ____,  # Hint: log_loss(y_test, y_proba)
}

print(f"\n=== Manual Evaluation ===")
for metric, value in eval_metrics.items():
    print(f"  {metric}: {value:.4f}")



### Checkpoint 3


In [ ]:
assert eval_metrics["auc_roc"] > 0.5, "Should beat random baseline"
print("\n✓ Checkpoint 3 passed — manual training and evaluation verified\n")



## TASK 4: Persist results with db.express


In [ ]:
if HAS_DATAFLOW:

    async def persist_results():
        await db.initialize()
        # TODO: Create a ModelEvaluation record using db.express.create
        # Hint: await db.express.create("ModelEvaluation", {...})
        eval_record = await db.express.create(
            ____,  # Hint: "ModelEvaluation"
            {
                "model_name": "lgbm_credit_v1",
                "dataset": "sg_credit_scoring",
                "accuracy": eval_metrics["accuracy"],
                "f1_score": eval_metrics["f1"],
                "auc_roc": eval_metrics["auc_roc"],
                "auc_pr": eval_metrics["auc_pr"],
                "log_loss_val": eval_metrics["log_loss"],
                "train_size": X_train.shape[0],
                "test_size": X_test.shape[0],
                "feature_count": X_train.shape[1],
                "hyperparameters": str(model.get_params()),
            },
        )
        print(f"\nPersisted evaluation: ID={eval_record['id']}")

        # TODO: Create a ModelArtifact record with model_name, version, artifact_path
        # Hint: await db.express.create("ModelArtifact", {...})
        artifact_record = await db.express.create(
            ____,  # Hint: "ModelArtifact"
            {
                "model_name": "lgbm_credit_v1",
                "version": 1,
                "artifact_path": "models/lgbm_credit_v1.pkl",
                "is_production": False,
                "created_by": "mlfp03_ex7",
            },
        )
        print(f"Persisted artifact: ID={artifact_record['id']}")
        return eval_record, artifact_record

    eval_record, artifact_record  = await persist_results()
else:
    eval_record = {"id": "skipped"}
    artifact_record = {"id": "skipped"}
    print("  DataFlow not available — skipping persistence.")



### Checkpoint 4


In [ ]:
assert eval_record is not None, "Evaluation record should exist"
assert "id" in eval_record, "Record should have auto-assigned ID"
print("\n✓ Checkpoint 4 passed — results persisted to DataFlow\n")



## TASK 5: Bayesian hyperparameter search


In [ ]:
# HyperparameterSearch uses Bayesian optimisation (surrogate model +
# acquisition function) to efficiently explore the hyperparameter space.

print(f"\n=== Hyperparameter Search ===")

if HAS_HYPERPARAM:
    # TODO: Define a SearchSpace with 5 ParamDistributions:
    #   n_estimators (int, 100-1000), learning_rate (float, 0.01-0.3, log=True),
    #   max_depth (int, 3-10), num_leaves (int, 15-127), min_child_samples (int, 5-50)
    search_space = SearchSpace(
        params=[
            ParamDistribution(
                ____, ____, low=____, high=____
            ),  # Hint: "n_estimators", "int", 100, 1000
            ParamDistribution(
                ____, ____, low=____, high=____, log=____
            ),  # Hint: "learning_rate", "float", 0.01, 0.3, True
            ParamDistribution("max_depth", "int", low=3, high=10),
            ParamDistribution("num_leaves", "int", low=15, high=127),
            ParamDistribution("min_child_samples", "int", low=5, high=50),
        ]
    )

    # TODO: Configure SearchConfig with n_trials=20, metric="average_precision",
    #       direction="maximize", cv_folds=5, random_state=42
    search_config = SearchConfig(
        n_trials=____,  # Hint: 20
        metric=____,  # Hint: "average_precision"
        direction=____,  # Hint: "maximize"
        cv_folds=5,
        random_state=42,
    )

    # TODO: Create HyperparameterSearch and call .search()
    # Hint: searcher = HyperparameterSearch(search_space, search_config)
    searcher = ____
    best_params, best_score, all_trials = searcher.search(
        estimator_class=____,  # Hint: lgb.LGBMClassifier
        X=X_train,
        y=y_train,
        fixed_params={"random_state": 42, "verbose": -1},
    )

    print(f"Best AUC-PR: {best_score:.4f}")
    print(f"Best params: {best_params}")
    print(f"\nTop 5 trials:")
    sorted_trials = sorted(all_trials, key=lambda t: t["score"], reverse=True)
    for i, trial in enumerate(sorted_trials[:5], 1):
        print(f"  {i}. score={trial['score']:.4f}, params={trial['params']}")
else:
    # Manual grid search as fallback
    print("Using manual grid search (HyperparameterSearch not available)...")
    param_grid = [
        {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 5},
        {"n_estimators": 500, "learning_rate": 0.1, "max_depth": 6},
        {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 7},
        {"n_estimators": 700, "learning_rate": 0.03, "max_depth": 8},
    ]

    best_score = -1
    best_params = param_grid[0]
    for params in param_grid:
        m = lgb.LGBMClassifier(
            **params,
            random_state=42,
            verbose=-1,
            scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
        )
        scores = cross_val_score(m, X_train, y_train, cv=5, scoring="average_precision")
        mean_score = scores.mean()
        print(f"  params={params}, AUC-PR={mean_score:.4f}")
        if mean_score > best_score:
            best_score = mean_score
            best_params = params

    print(f"\nBest: AUC-PR={best_score:.4f}, params={best_params}")

# Train final model with best hyperparameters
best_model = lgb.LGBMClassifier(
    **best_params,
    random_state=42,
    verbose=-1,
    scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
)
best_model.fit(X_train, y_train)
y_proba_best = best_model.predict_proba(X_test)[:, 1]
best_auc_pr = average_precision_score(y_test, y_proba_best)
print(f"Final model AUC-PR: {best_auc_pr:.4f}")



### Checkpoint 5


In [ ]:
assert best_score > 0, "Best score should be positive"
assert best_params is not None, "Should find best parameters"
# INTERPRETATION: Bayesian optimisation models the relationship between
# hyperparameters and performance, then intelligently chooses the next
# point to evaluate.  20 Bayesian trials can match 200 random trials.
print("\n✓ Checkpoint 5 passed — hyperparameter search complete\n")



## TASK 6: Branching workflow with conditional execution


In [ ]:
# Real ML pipelines need branching: if model performance exceeds a
# threshold, register it; otherwise, trigger retraining with different
# hyperparameters.

branching_workflow = WorkflowBuilder("credit_scoring_branching")

branching_workflow.add_node(
    "DataPreprocessNode",
    "preprocess",
    {"data_source": "sg_credit_scoring", "target": "default"},
)

branching_workflow.add_node(
    "ModelTrainNode",
    "train_primary",
    {"model_class": "lightgbm.LGBMClassifier", "hyperparameters": best_params},
    connections=["preprocess"],
)

branching_workflow.add_node(
    "ModelEvalNode",
    "evaluate",
    {"metrics": ["auc_pr", "brier_score"]},
    connections=["train_primary"],
)

# TODO: Add a ConditionalNode named "quality_gate" that branches on auc_pr > 0.5
# Hint: "ConditionalNode", condition="auc_pr > 0.5", true_output="register",
#       false_output="retrain", connections=["evaluate"]
branching_workflow.add_node(
    ____,  # Hint: "ConditionalNode"
    "quality_gate",
    {
        "condition": ____,  # Hint: "auc_pr > 0.5"
        "true_output": ____,  # Hint: "register"
        "false_output": "retrain",
    },
    connections=["evaluate"],
)

branching_workflow.add_node(
    "PersistNode",
    "register",
    {"storage": "sqlite:///mlfp03_models.db", "stage": "staging"},
    connections=["quality_gate"],
)

print("\n=== Branching Workflow ===")
print("Pipeline: preprocess → train → evaluate → quality_gate → register/retrain")
print("Quality gate: AUC-PR > 0.5 → register model, else → retrain")

try:
    branch_results, branch_run_id = runtime.execute(branching_workflow.build())
    print(f"Branching workflow executed: {branch_run_id}")
except Exception as e:
    print(f"  Branching workflow demo (ConditionalNode requires registration)")
    branch_results, branch_run_id = {}, "branch-demo"



### Checkpoint 6


In [ ]:
print("\n✓ Checkpoint 6 passed — branching workflow defined\n")



## TASK 7: Define ModelSignature


In [ ]:
# TODO: Create a FeatureSchema named "credit_model_input" using FeatureField
# Hint: FeatureSchema(name=..., features=[FeatureField(name=f, dtype="float64") for f in ...],
#       entity_id_column="application_id")
input_schema = FeatureSchema(
    name=____,  # Hint: "credit_model_input"
    features=[
        FeatureField(name=f, dtype="float64")
        for f in ____  # Hint: col_info["feature_columns"]
    ],
    entity_id_column="application_id",
)

# TODO: Create a ModelSignature with input_schema, output_columns, output_dtypes, model_type
# Hint: ModelSignature(input_schema=..., output_columns=["default_probability", "default_label"],
#       output_dtypes=["float64", "int64"], model_type="classifier")
signature = ModelSignature(
    input_schema=____,  # Hint: the input_schema you just created
    output_columns=____,  # Hint: ["default_probability", "default_label"]
    output_dtypes=["float64", "int64"],
    model_type=____,  # Hint: "classifier"
)

print(f"\n=== ModelSignature ===")
print(f"Input features: {len(signature.input_schema.features)}")
print(f"Output: {signature.output_columns}")
print(f"Model type: {signature.model_type}")



### Checkpoint 7


In [ ]:
assert len(signature.input_schema.features) == len(
    col_info["feature_columns"]
), "Signature features should match training features"
assert "default_probability" in signature.output_columns
print("\n✓ Checkpoint 7 passed — ModelSignature validated\n")



## TASK 8: Register model in ModelRegistry


In [ ]:
async def register_model():
    # TODO: Create a ConnectionManager and initialize it
    # Hint: conn = ConnectionManager("sqlite:///mlfp03_models.db"); await conn.initialize()
    conn = ____  # Hint: ConnectionManager("sqlite:///mlfp03_models.db")
    await conn.initialize()

    model_id = None
    if HAS_REGISTRY:
        # TODO: Create a ModelRegistry and call register_model
        # Hint: registry = ModelRegistry(conn)
        registry = ____
        model_bytes = pickle.dumps(best_model)

        if HAS_METRIC_SPEC:
            metrics_list = [
                MetricSpec(name="auc_pr", value=best_auc_pr),
                MetricSpec(name="auc_roc", value=roc_auc_score(y_test, y_proba_best)),
            ]
        else:
            metrics_list = []

        # TODO: Register the model with name, artifact (model_bytes), and metrics
        # Hint: await registry.register_model(name="credit_default_v2", artifact=..., metrics=...)
        model_version = await registry.register_model(
            name=____,  # Hint: "credit_default_v2"
            artifact=____,  # Hint: model_bytes
            metrics=metrics_list,
        )
        model_id = model_version.version
        print(f"\nModel registered: credit_default_v2 v{model_id}")
    else:
        model_id = "simulated-v1"
        print(f"\nModel registration simulated: {model_id}")

    await conn.close()
    return model_id


model_version_id  = await register_model()



### Checkpoint 8


In [ ]:
assert model_version_id is not None, "Model should be registered"
print("\n✓ Checkpoint 8 passed — model registered in ModelRegistry\n")



## TASK 9: Promote model through lifecycle


In [ ]:
async def promote_model():
    conn = ConnectionManager("sqlite:///mlfp03_models.db")
    await conn.initialize()

    if HAS_REGISTRY:
        registry = ModelRegistry(conn)
        # TODO: Promote the model to "production" stage with a reason string
        # Hint: await registry.promote_model(name=..., version=..., target_stage=..., reason=...)
        await registry.promote_model(
            name=____,  # Hint: "credit_default_v2"
            version=____,  # Hint: model_version_id
            target_stage=____,  # Hint: "production"
            reason=(
                f"Passed quality gates: AUC-PR={best_auc_pr:.4f}, "
                f"hyperparams optimised via {'Bayesian' if HAS_HYPERPARAM else 'grid'} search"
            ),
        )
        print(f"\nModel promoted to production: credit_default_v2 v{model_version_id}")
        print(f"Reason logged for audit trail")
    else:
        print(f"\nModel promotion simulated (ModelRegistry not available)")

    await conn.close()


await promote_model()

print("\nModel lifecycle:")
print("  experiment → register (staging) → promote (production) → retire")
print("  Every transition is logged with a reason for the audit trail.")



### Checkpoint 9


In [ ]:
print("\n✓ Checkpoint 9 passed — model promoted to production\n")



## TASK 10: Query persisted results


In [ ]:
if HAS_DATAFLOW:

    async def query_results():
        # TODO: List all ModelEvaluation records using db.express.list
        # Hint: await db.express.list("ModelEvaluation")
        evals = await db.express.list(____)  # Hint: "ModelEvaluation"
        print(f"\n=== Persisted Evaluations ({len(evals)}) ===")
        for e in evals:
            print(
                f"  {e['model_name']}: AUC-ROC={e['auc_roc']:.4f}, "
                f"AUC-PR={e['auc_pr']:.4f}"
            )

        artifacts = await db.express.list("ModelArtifact")
        print(f"\nModel Artifacts ({len(artifacts)}):")
        for a in artifacts:
            status = "PRODUCTION" if a["is_production"] else "staging"
            print(f"  {a['model_name']} v{a['version']}: {status}")

        await db.close()
        return evals, artifacts

    query_evals, query_artifacts  = await query_results()
else:
    query_evals = [{"auc_roc": 0.0, "model_name": "skipped"}]
    query_artifacts = [{"model_name": "skipped", "version": 0, "is_production": False}]



### Checkpoint 10


In [ ]:
assert len(query_evals) > 0, "Should have at least one evaluation"
print("\n✓ Checkpoint 10 passed — persisted results queried\n")



## TASK 11: Complete orchestrated pipeline


The full M3 pipeline orchestrated as a Kailash workflow:

  ┌──────────────┐    ┌───────────────┐    ┌──────────────┐
  │  Preprocess   │ →  │  Hyperparam   │ →  │    Train     │
  │  (Pipeline)   │    │  (Bayesian)   │    │  (LightGBM)  │
  └──────────────┘    └───────────────┘    └──────┬───────┘
                                                   │
  ┌──────────────┐    ┌───────────────┐    ┌──────▼───────┐
  │  Persist     │ ←  │  Quality Gate │ ←  │   Evaluate   │
  │  (DataFlow)  │    │ (AUC-PR>0.5) │    │  (Metrics)   │
  └──────┬───────┘    └───────────────┘    └──────────────┘
         │
  ┌──────▼───────┐    ┌───────────────┐
  │  Register    │ →  │   Promote     │
  │  (Registry)  │    │ (→Production) │
  └──────────────┘    └───────────────┘

Every step is: named, versioned, reproducible, and auditable.


In [ ]:
print(f"\n=== Complete Orchestrated Pipeline ===")
print(
)



### Checkpoint 11


In [ ]:
print("\n✓ Checkpoint 11 passed — complete pipeline documented\n")



## TASK 12: Pipeline audit and reproducibility check


In [ ]:
print(f"\n=== Pipeline Audit ===")
print(f"Data source: sg_credit_scoring.parquet")
print(f"Preprocessing: ordinal encoding, median imputation, no normalisation")
print(f"Train/test split: 80/20, seed=42")
print(
    f"Model: LightGBM with {'Bayesian' if HAS_HYPERPARAM else 'grid'} optimised hyperparameters"
)
print(f"Best hyperparameters: {best_params}")
print(f"Evaluation metrics:")
for k, v in eval_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"Model version: {model_version_id}")
print(f"Storage: sqlite:///mlfp03_models.db")

# Reproducibility: verify same seed → same results
model_repro = lgb.LGBMClassifier(
    **best_params,
    random_state=42,
    verbose=-1,
    scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
)
model_repro.fit(X_train, y_train)
y_proba_repro = model_repro.predict_proba(X_test)[:, 1]
auc_repro = average_precision_score(y_test, y_proba_repro)
print(
    f"\nReproducibility check: AUC-PR = {auc_repro:.4f} (original: {best_auc_pr:.4f})"
)
assert abs(auc_repro - best_auc_pr) < 0.001, "Same seed should reproduce same results"
print("  ✓ Reproducible: same seed → same results")



### Checkpoint 12


In [ ]:
print("\n✓ Checkpoint 12 passed — pipeline audit and reproducibility verified\n")



## REFLECTION


✓ WorkflowBuilder: declare ML steps as named nodes with connections
  ✓ LocalRuntime.execute(workflow.build()): the canonical execution pattern
  ✓ Branching workflows: conditional execution based on quality gates
  ✓ HyperparameterSearch: Bayesian optimisation of model hyperparameters
  ✓ DataFlow @db.model: declarative schema → automatic table creation
  ✓ db.express.create/list: async CRUD without writing SQL
  ✓ ModelSignature: input/output contract that InferenceServer enforces
  ✓ ModelRegistry: register → stage → promote lifecycle
  ✓ Persistence: every evaluation stored, every artifact tracked
  ✓ Reproducibility: same seed + same config → same results

  KEY INSIGHT: The workflow is not just automation — it is documentation.
  When you encode your pipeline as a DAG, every step is named, versioned,
  and reproducible.  Six months later you can re-run the exact same
  pipeline with different data and get comparable results.

  THE PATTERNS:
    workflow = WorkflowBuilder("name")
    workflow.add_node("NodeType", "name", config, connections=[...])
    results, run_id = runtime.execute(workflow.build())

    await db.express.create("Model", data)
    await registry.register_model(name, artifact, metrics)
    await registry.promote_model(name, version, stage, reason)

  NEXT: Exercise 8 brings everything together — the complete production
  pipeline including conformal prediction, DriftMonitor, model card
  generation, and a full monitoring dashboard.


In [ ]:
print("═" * 70)
print("  WHAT YOU'VE MASTERED")
print("═" * 70)
print(
)
print("═" * 70)

